### Change directory:

In [1]:
import os

In [2]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer\\research'

In [3]:
# Change directory:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer'

### 1. Update the config.yaml 

We update config.yaml to add data validation settings, which create a data_validation folder inside artifacts to store validation outputs like status.txt. This file records whether all required dataset files (defined in All_Required_Files) are present, returning True if complete and False otherwise.

In [5]:
# data validation configuration section:
# data_validation:
#   root_dir: artifacts/data_validation # Folder where all validation outputs will be stored
#   STATUS_FILE: artifacts/data_validation/status.txt # Path of the file that will store validation result (True or False)
#   ALL_REQUIRED_FILES: ["train", "test", "validation"] # List of required dataset files that must exist before training

### 2. Update the params.py

In this case, we dont have any parameters. Whenever we define the model configuration, we usually include parameters, but at this stage, they are not required.

### 3. Update/Create the Entity:

In [6]:
from dataclasses import dataclass # Create simple classes for storing data.
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig: # Defines a class to store all configuration settings for data validation.
    root_dir: Path # Folder path where validation files will be stored (get path from config.yaml file)
    STATUS_FILE: str # Path of the file that stores validation result (True/False)
    ALL_REQUIRED_FILES: list # List of all required dataset files that must exist for training.

### 4. Update the Configuration Manager:

In [7]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig: # Defines a function that returns a DataValidationConfig object.
        config = self.config.data_validation # Gets the data validation settings from the main config.

        create_directories([config.root_dir]) # Creates the validation folder if it does not already exist.
        #Creates an object of DataValidationConfig:
        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir, # Sets the folder path for validation outputs.
            STATUS_FILE=config.STATUS_FILE, # Sets the path for the status file (True/False result).
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES, # Passes the list of required dataset files.
        )

        return data_validation_config # Returns the final configuration object so it can be used in the pipeline

### 5. Update the Components:

In [9]:
import os
from textsummarizer.logging import logger

In [10]:
# Creates a class for data validation logic:
class DataValiadtion:
    def __init__(self, config: DataValidationConfig): # Constructor that receives configuration object.
        self.config = config # Stores config inside the class for later use.


    # Function that checks if all required files exist and returns True/False.
    def validate_all_files_exist(self)-> bool:
        try:
            validation_status = None # Initializes validation status (not decided yet).
             
            all_files = os.listdir(os.path.join("artifacts","data_ingestion","samsumdata","samsum_dataset")) # Gets list of files inside the dataset folder.
 
            for file in all_files:
                if file not in self.config.ALL_REQUIRED_FILES: # Checks if a file is missing from required list.
                    validation_status = False # If any required file is missing → validation fails.
                    with open(self.config.STATUS_FILE, 'w') as f: # Writes validation result into status.txt.
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True # If file exists → mark validation as successful.
                    with open(self.config.STATUS_FILE, 'w') as f: # Saves success status into file.
                        f.write(f"Validation status: {validation_status}")

            return validation_status # Returns final result (True or False).
        
        except Exception as e: # If anything breaks, show the error.
            raise e

### 6. Update the Pipelines:

In [11]:
try:
    config = ConfigurationManager() # Creates configuration manager (loads YAML + params).
    data_validation_config = config.get_data_validation_config() # Gets data validation settings from config.
    data_validation = DataValiadtion(config=data_validation_config) # Creates DataValidation object using config.
    data_validation.validate_all_files_exist() # Runs validation to check if required dataset files exist.
except Exception as e:
    raise e

[2026-05-13 15:01:32,066: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-13 15:01:32,071: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-13 15:01:32,073: INFO: common: created directory at: artifacts]
[2026-05-13 15:01:32,075: INFO: common: created directory at: artifacts/data_validation]
